# RoboTwin × LingBot-VLA-v2 on ROCm

This notebook is the interactive entry point for the `external-data` image. Run cells selectively: model serving and evaluation are long-running GPU workloads. The platform-provided data must be available at `/models/robotwin-persistent`. Runtime outputs are written to `/workspace/runtime`.

In [ ]:
from pathlib import Path
import os, signal, subprocess, time, urllib.request

ROBOTWIN = Path('/RoboTwin')
RUNTIME = Path('/workspace/runtime')
PYTHON = '/opt/robotwin-env/bin/python'
MODEL = ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/models/robbyant_lingbot-vla-v2-6b-robotwin/checkpoints/global_step_50000/hf_ckpt'
RUNTIME.joinpath('outputs/logs').mkdir(parents=True, exist_ok=True)
print('RoboTwin:', ROBOTWIN)
print('Model:', MODEL)

## 1. Environment and mounted-data check

In [ ]:
required = [
    ROBOTWIN / 'assets/objects/objaverse/list.json',
    ROBOTWIN / 'data/demo_clean',
    ROBOTWIN / 'data/lerobot',
    MODEL / 'model.safetensors.index.json',
]
for path in required:
    assert path.exists(), f'Missing: {path}'
subprocess.run([PYTHON, '-c', "import torch; print('torch=', torch.__version__, 'hip=', torch.version.hip, 'gpu=', torch.cuda.is_available(), 'count=', torch.cuda.device_count())"], check=True)

### Expected svulkan2 warning on AMD

During SAPIEN initialization, the following messages are expected on ROCm/AMD and can be ignored if Vulkan rendering and evaluation continue:

```text
[svulkan2] [error] CUDA Error: cudaErrorInsufficientDriver
[svulkan2] [error] Failed to initialize denoiser
```

svulkan2 is probing its optional NVIDIA CUDA denoiser; RoboTwin continues through the Vulkan renderer. Investigate `/dev/dri`, `PYOPENGL_PLATFORM=egl`, and `vulkaninfo --summary` only if the process subsequently exits or camera images are missing.

## 2. Start the model server on port 13400

Stop any existing process using port 13400 before running this cell. The server log is written under `/workspace/runtime/outputs/logs`.

In [ ]:
server_log = RUNTIME / 'outputs/logs/official_server.log'
server_command = [
    'bash', str(ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/scripts/launch_official_server.sh'),
    '0', '13400', str(server_log), 'False', str(MODEL),
]
server_log_handle = server_log.open('ab')
SERVER_PROCESS = subprocess.Popen(server_command, cwd=ROBOTWIN, stdout=server_log_handle, stderr=subprocess.STDOUT, start_new_session=True)
print('server pid:', SERVER_PROCESS.pid, 'log:', server_log)
for _ in range(600):
    if SERVER_PROCESS.poll() is not None:
        raise RuntimeError(f'Server exited with code {SERVER_PROCESS.returncode}; inspect {server_log}')
    try:
        urllib.request.urlopen('http://127.0.0.1:13400/healthz', timeout=2).read()
        print('server ready: 127.0.0.1:13400')
        break
    except Exception:
        time.sleep(2)
else:
    raise TimeoutError(f'Server did not become ready; inspect {server_log}')

## 3. Run 10 closed-loop `adjust_bottle` episodes

> **重要提示：下面两行不是评测失败，可以忽略。** svulkan2 正在探测仅供 NVIDIA 使用的可选 CUDA denoiser；在 AMD ROCm 环境中出现它们是正常现象，RoboTwin 会继续使用 Vulkan 渲染：
>
> ```text
> [2026-08-26 12:03:21.547] [svulkan2] [error] CUDA Error: cudaErrorInsufficientDriver
> [2026-08-26 12:03:21.547] [svulkan2] [error] Failed to initialize denoiser
> ```
> 只有评测进程随后退出或相机图像缺失时，才需要检查 Vulkan 和设备配置。

In [ ]:
EVAL_EPISODES = 10
eval_command = [
    PYTHON, str(ROBOTWIN / 'scripts/eval_policy_xpolicylab.py'),
    '--task_name', 'adjust_bottle', '--task_config', 'demo_clean',
    '--policy_name', 'LingBot-VLA-v2', '--protocol', 'lingbot_vla_v2',
    '--host', '127.0.0.1', '--port', '13400', '--device_id', '0',
    '--seed', '0', '--test_num', str(EVAL_EPISODES), '--expert_check', 'false',
    '--eval_batch', 'false',
]
eval_env = os.environ.copy()
eval_env.update(ROBOTWIN_DISABLE_CUROBO='1', ROBOTWIN_EE_PLANNER='mplib', PYOPENGL_PLATFORM='egl')
base_eval_started = time.perf_counter()
subprocess.run(eval_command, cwd=ROBOTWIN, env=eval_env, check=True)
base_eval_elapsed = time.perf_counter() - base_eval_started
print(f'base evaluation total: {base_eval_elapsed:.2f}s ({base_eval_elapsed / 60:.2f} min)')
print(f'base evaluation average per episode: {base_eval_elapsed / EVAL_EPISODES:.2f}s')

## 4. LoRA fine-tuning

Choose `GPU_COUNT` as 1, 2, or 4 and set `TRAIN_STEPS` explicitly. The default 100 steps is a smoke test for the complete pipeline, not an expectation of useful fine-tuning quality. Start with 1,000–5,000 steps for an experiment and select the final value from held-out closed-loop evaluation. The effective global batch size remains 4. This cell stops the model server started above before training so it releases GPU memory.

In [ ]:
if 'SERVER_PROCESS' in globals() and SERVER_PROCESS.poll() is None:
    os.killpg(os.getpgid(SERVER_PROCESS.pid), signal.SIGTERM)
    SERVER_PROCESS.wait(timeout=30)
    server_log_handle.close()
    print('model server stopped before training')

GPU_COUNT = 1  # Supported values: 1, 2, 4
TRAIN_STEPS = 100  # Smoke test; increase to e.g. 1000 or 5000 for fine-tuning
if GPU_COUNT not in (1, 2, 4):
    raise ValueError('GPU_COUNT must be 1, 2, or 4')
if TRAIN_STEPS <= 0:
    raise ValueError('TRAIN_STEPS must be positive')
gradient_accumulation_steps = 4 // GPU_COUNT
training_root = ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin'
source_root = training_root / 'source/lingbot-vla-v2'
training_yaml = training_root / 'training/reproduction_100steps/lingbotvla_cli.yaml'
training_output = RUNTIME / f'outputs/reproduction_{TRAIN_STEPS}steps'
train_command = [
    PYTHON, '-m', 'torch.distributed.run', '--standalone',
    f'--nproc-per-node={GPU_COUNT}', '-m', 'tasks.vla.train_lingbotvla',
    str(training_yaml),
    '--train.data_parallel_shard_size', str(GPU_COUNT),
    '--train.gradient_accumulation_steps', str(gradient_accumulation_steps),
    '--train.max_steps', str(TRAIN_STEPS),
    '--train.save_steps', str(TRAIN_STEPS),
    '--train.output_dir', str(training_output),
]
train_env = os.environ.copy()
train_env['HIP_VISIBLE_DEVICES'] = ','.join(str(i) for i in range(GPU_COUNT))
train_env.pop('ROCR_VISIBLE_DEVICES', None)
train_env.pop('CUDA_VISIBLE_DEVICES', None)
training_started = time.perf_counter()
subprocess.run(train_command, cwd=source_root, env=train_env, check=True)
training_elapsed = time.perf_counter() - training_started
print(f'LoRA training total: {training_elapsed:.2f}s ({training_elapsed / 60:.2f} min, {training_elapsed / 3600:.2f} h)')
print(f'LoRA wall-clock average per optimizer step: {training_elapsed / TRAIN_STEPS:.2f}s')

## 5. Merge the LoRA checkpoint

In [ ]:
checkpoint = training_output / f'checkpoints/global_step_{TRAIN_STEPS}'
MERGED = training_output / f'merged_checkpoint/global_step_{TRAIN_STEPS}/hf_ckpt'
assert checkpoint.exists(), f'Missing checkpoint: {checkpoint}'
merge_command = [
    PYTHON, str(ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/scripts/merge_lora_dcp.py'),
    '--checkpoint', str(checkpoint),
    '--training-output', str(training_output),
    '--base-model', str(MODEL), '--output', str(MERGED),
    '--rank', '8', '--alpha', '16',
]
subprocess.run(merge_command, cwd=ROBOTWIN, check=True)
assert (MERGED / 'model.safetensors.index.json').exists(), f'Merge output is incomplete: {MERGED}'
print('merged model:', MERGED)

## 6. Start the merged model on port 13400

The original server was stopped before training, so the merged model reuses the same endpoint.

In [ ]:
merged_log = RUNTIME / 'outputs/logs/merged_server.log'
merged_server_command = [
    'bash', str(ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/scripts/launch_official_server.sh'),
    '0', '13400', str(merged_log), 'False', str(MERGED),
]
merged_log_handle = merged_log.open('ab')
MERGED_SERVER_PROCESS = subprocess.Popen(merged_server_command, cwd=ROBOTWIN, stdout=merged_log_handle, stderr=subprocess.STDOUT, start_new_session=True)
print('merged server pid:', MERGED_SERVER_PROCESS.pid, 'log:', merged_log)
for _ in range(600):
    if MERGED_SERVER_PROCESS.poll() is not None:
        raise RuntimeError(f'Merged server exited with code {MERGED_SERVER_PROCESS.returncode}; inspect {merged_log}')
    try:
        urllib.request.urlopen('http://127.0.0.1:13400/healthz', timeout=2).read()
        print('merged server ready: 127.0.0.1:13400')
        break
    except Exception:
        time.sleep(2)
else:
    raise TimeoutError(f'Merged server did not become ready; inspect {merged_log}')

## 7. Validate the merged model with 10 closed-loop episodes

In [ ]:
merged_eval_started = time.perf_counter()
subprocess.run(eval_command, cwd=ROBOTWIN, env=eval_env, check=True)
merged_eval_elapsed = time.perf_counter() - merged_eval_started
print(f'merged evaluation total: {merged_eval_elapsed:.2f}s ({merged_eval_elapsed / 60:.2f} min)')
print(f'merged evaluation average per episode: {merged_eval_elapsed / EVAL_EPISODES:.2f}s')

## 8. Optional: stop the merged model server

In [ ]:
if 'MERGED_SERVER_PROCESS' in globals() and MERGED_SERVER_PROCESS.poll() is None:
    os.killpg(os.getpgid(MERGED_SERVER_PROCESS.pid), signal.SIGTERM)
    MERGED_SERVER_PROCESS.wait(timeout=30)
    merged_log_handle.close()
    print('merged model server stopped')
else:
    print('no merged server started by this notebook kernel')